Численный эксперимент
  
Целевая метрика — средняя выручка с клиента за время эксперимента. Целевую метрику считаем на неделе с 21 по 28 марта. 
Уровень значимости — 0.05. 
Критерий — тест Стьюдента. 
Размер групп — 1000. 
Ожидаемый эффект — средняя выручка увеличится на 10%.

Нужно оценить вероятности ошибок II рода для трёх вариантов добавления эффекта:

1. Добавление константы ко всем значениям;

2. Умножение на константу всех значений;

3. Добавление константы к 2.5% значений.

Для решения используйте данные из файла 2022-04-01T12_df_sales.csv.

Ответ: 132

In [9]:
import time
import os

import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import norm, ttest_ind
from datetime import datetime
from tqdm.notebook import tqdm

In [7]:
URL_BASE = 'https://raw.githubusercontent.com/ab-courses/simulator-ab-datasets/main/2022-04-01/'

def read_database(file_name):
    return pd.read_csv(os.path.join(URL_BASE, file_name))

df_sales = read_database('2022-04-01T12_df_sales.csv')
df_sales['date'] = pd.to_datetime(df_sales['date'])
df_sales.head(3)


,sale_id,date,count_pizza,count_drink,price,user_id
0,1000001,2022-02-04 10:00:24,1,0,720,1c1543
1,1000002,2022-02-04 10:02:28,1,1,930,a9a6e8
2,1000003,2022-02-04 10:02:35,3,1,1980,23420a


In [11]:
begin_date = datetime(2022, 3, 21)
end_date = datetime(2022, 3, 28)

# пользователи, совершавшие покупки до 28 марта
df_users = df_sales[df_sales['date'] < end_date][['user_id']].drop_duplicates()

# метрика: суммарная выручка пользователя за 21–28 марта
df_metrics = (
    df_sales
    [(df_sales['date'] >= begin_date) & (df_sales['date'] < end_date)]
    .groupby('user_id')[['price']].sum()
    .reset_index()  
)
df = pd.merge(df_users, df_metrics, on='user_id', how='left').fillna(0)

#_________________________________________________________________

alpha = 0.05
sample_size = 1000
effect = 0.1

pvalues = {'one': [], 'two': [], 'three': []}
values = df['price'].values

# средняя выручка на исторических данных
mean_ = values.mean()

In [15]:
for _ in tqdm(range(30000)):      # tqdm -  процесс выполнения
    # выбираем случайные группы
    a, b = np.random.choice(values, (2, sample_size,), False)

    # добавляем эффект тремя способами
    # 1) добавление константы ко всем значениям
    b_one = b + mean_ * effect 

    # 2) умножение на константу всех значений
    b_two = b * (1 + effect)
    indexes = np.random.choice(np.arange(sample_size), int(sample_size * 0.025), False)
    add_value = effect * mean_ * sample_size / len(indexes)
    mask = np.zeros(sample_size)
    mask[indexes] += 1

    # 3) добавление константы к 2.5% значений
    b_three = b + mask * add_value

    
    # считаем и сохраняем p-value
    for b_, key in ((b_one, 'one',), (b_two, 'two',), (b_three, 'three',),):
        pvalues[key].append(stats.ttest_ind(a, b_).pvalue)

# считаем точечные оценки вероятностей ошибки II рода
for key, v in pvalues.items():
    errors = (np.array(v) > alpha).astype(int)
    part_errors = np.mean(errors)
    print(f'{key}: part errors = {part_errors:0.4f}')

print('_________________________________________________________________')


# проверим, что отличия статистически значимые
print(stats.ttest_ind(pvalues['one'], pvalues['three']).pvalue)
print(stats.ttest_ind(pvalues['two'], pvalues['three']).pvalue)

  0%|          | 0/30000 [00:00<?, ?it/s]

one: part errors = 0.8147
two: part errors = 0.8303
three: part errors = 0.8260
_________________________________________________________________
0.00013803958944629696
0.0007797203640831


  0%|          | 0/30000 [00:00<?, ?it/s]

one: part errors = 0.8133
two: part errors = 0.8290
three: part errors = 0.8253
0.006686978253998567
0.01635847615392853
